---
# 01. EDA — 채용 플랫폼 로그 데이터 탐색
---


채용 플랫폼의 2022~2023 사용자 로그를 탐색하고, 테이블 구조·기간·유저 수·URL 패턴을 확인한다. 원본 사용자 식별자가 노출되는 출력과 DB 계정 정보는 저장소에 포함하지 않는다.

## 1. 환경 설정 & DB 연결 확인
---

### 1-1. 라이브러리 환경 설정

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import koreanize_matplotlib

# notebooks/에서 실행해도 프로젝트 루트의 src 모듈을 불러올 수 있게 설정
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import get_engine, load_logs

warnings.filterwarnings("ignore")


### 1-2. DB 연결

In [ ]:
engine = get_engine()

# 연결 확인
pd.read_sql("SELECT 1 AS ok", engine)


### 1-3. 테이블 목록

In [ ]:
query = """
SHOW tables;
"""
pd.read_sql(query, engine)

,Tables_in_job_subs
0,application
1,com_2022
2,com_2023
3,company
4,companyaddress
5,companyfund
6,job
7,jobaddress
8,jobbookmark
9,log_2022


In [ ]:
# 전체 테이블 컬럼 한번에 확인
tables = ['Application', 'Company', 'CompanyAddress', 'CompanyFund',
          'Job', 'JobAddress', 'JobBookmark', 'com_2022', 'com_2023']

for t in tables:
    print(f"\n=== {t} ===")
    display(pd.read_sql(f"DESCRIBE {t};", engine))


=== Application ===


,Field,Type,Null,Key,Default,Extra
0,cdate,datetime,YES,,None,
1,company_uuid,varchar(50),YES,,None,
2,job_uuid,varchar(50),YES,,None,
3,user_uuid,varchar(50),YES,,None,
4,application_uuid,varchar(50),YES,,None,



=== Company ===


,Field,Type,Null,Key,Default,Extra
0,cdate,datetime,YES,,None,
1,mdate,datetime,YES,,None,
2,found_date,varchar(50),YES,,None,
3,employee_count,varchar(50),YES,,None,
4,view_count,int,YES,,None,
5,follow_count,int,YES,,None,
6,reference_count,int,YES,,None,
7,company_uuid,varchar(50),YES,,None,



=== CompanyAddress ===


...

,Field,Type,Null,Key,Default,Extra
0,cdate,datetime,YES,,None,
1,job_uuid,varchar(50),YES,,None,
2,user_uuid,varchar(50),YES,,None,



=== com_2022 ===


,Field,Type,Null,Key,Default,Extra
0,user_uuid,varchar(50),YES,,None,
1,URL,longtext,YES,,None,
2,timestamp,datetime,YES,,None,
3,date,varchar(50),YES,,None,
4,response_code,varchar(50),YES,,None,
5,method,varchar(50),YES,,None,



=== com_2023 ===


,Field,Type,Null,Key,Default,Extra
0,user_uuid,varchar(50),YES,,None,
1,URL,longtext,YES,,None,
2,timestamp,datetime,YES,,None,
3,date,varchar(50),YES,,None,
4,response_code,varchar(50),YES,,None,
5,method,varchar(50),YES,,None,


### 1-4. 각 테이블 행 수 확인

In [ ]:
query = """
SELECT '2022' AS log_year, COUNT(*) AS row_cnt FROM com_2022
UNION ALL
SELECT '2023', COUNT(*) FROM com_2023;
"""

pd.read_sql(query, engine)

,log_year,row_cnt
0,2022,9689040
1,2023,6914234


## 2. 데이터 로드 & 기본 탐색
---

### 2-1. 두 테이블 합치기 (com_2022 / com_2023)

- com_2022, log_2023 UNION ALL
- 에러 응답(400, 404, 500 등)은 정상 유저 행동이 아니므로 제외
- response_code 200, 302만 포함

In [ ]:
df = load_logs(engine)

In [ ]:
df.shape

(16468531, 6)

In [ ]:
# df.head()  # 원본 로그라 여기선 생략함

### 2-2. 기본 정보 확인

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16468531 entries, 0 to 16468530
Data columns (total 6 columns):
 #   Column         Dtype              
---  ------         -----              
 0   user_uuid      str                
 1   URL            str                
 2   timestamp      datetime64[us, UTC]
 3   date           datetime64[us]     
 4   response_code  str                
 5   method         str                
dtypes: datetime64[us, UTC](1), datetime64[us](1), str(4)
memory usage: 753.9 MB


### 2-3. 기간 & 유저 수 확인

In [ ]:
print(f"로그 기간: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"유니크 유저수: {df['user_uuid'].nunique():,}명")
print()
print(df['date'].dt.year.value_counts().sort_index())

로그 기간: 2022-01-01 ~ 2023-12-31
유니크 유저수: 21,340명

date
2022    9606067
2023    6862464
Name: count, dtype: int64


### 2-4. URL 파악

- URL 컬럼의 구조를 파악 -> 파싱 전략 수립
- 실제 분석은 전체 데이터 기준으로 진행 예정

In [ ]:
# 유니크 URL 전체 패턴 파악
url_patterns = (
    df['URL']
    .str.split('?').str[0]
    .value_counts()
    .reset_index()
)
url_patterns.columns = ['url_path', 'count']
print(f"URL 패턴 수: {len(url_patterns):,}")
url_patterns.head(50)

URL 패턴 수: 245


,url_path,count
0,api/jobs/job_title,1226738
1,jobs/id/id_title,1192107
2,api/users/id/template,1156004
3,api/jobs/id/other_jobs,1058774
4,@user_id,1056957
5,api/recommend_specialty,743424
6,suggest,636249
7,companies/company_id,629549
8,api/companies/id/view,615226
9,jobs,474610


In [ ]:
# - Acquistion

# - Activation

# - Retention

In [ ]:
step3_done = df[df['URL'].str.contains('signup/step3/done', case=False, na=False)]
print(f"signup/step3/done 고유 유저 수: {step3_done['user_uuid'].nunique():,}")

signup/step3/done 고유 유저 수: 4,746
